# Marketing ML Project

## Objectives
* Build a classifier that predicts wheter a person makes less than $50k or more than $50k given 40 demographic and employment related variables
* Create a segmentation model and demonstrate how the resulting groups differ from one another and how your client can use this model for marketing

In [12]:
import pandas as pd
import numpy as np

## Import dataset

In [13]:
# read column headers into a list
with open('TakeHomeProject/census-bureau.columns', 'r') as f:
    columns = [line.strip() for line in f]

# import main data
df = pd.read_csv('TakeHomeProject/census-bureau.data', header=None, names=headers)

# display first few rows
df.head()

,age,class of worker,detailed industry recode,detailed occupation recode,education,wage per hour,enroll in edu inst last wk,marital stat,major industry code,major occupation code,...,country of birth father,country of birth mother,country of birth self,citizenship,own business or self employed,fill inc questionnaire for veteran's admin,veterans benefits,weeks worked in year,year,label
0,73,Not in universe,0,0,High school graduate,0,Not in universe,Widowed,Not in universe or children,Not in universe,...,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,2,0,95,- 50000.
1,58,Self-employed-not incorporated,4,34,Some college but no degree,0,Not in universe,Divorced,Construction,Precision production craft & repair,...,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,2,52,94,- 50000.
2,18,Not in universe,0,0,10th grade,0,High school,Never married,Not in universe or children,Not in universe,...,Vietnam,Vietnam,Vietnam,Foreign born- Not a citizen of U S,0,Not in universe,2,0,95,- 50000.
3,9,Not in universe,0,0,Children,0,Not in universe,Never married,Not in universe or children,Not in universe,...,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,0,0,94,- 50000.
4,10,Not in universe,0,0,Children,0,Not in universe,Never married,Not in universe or children,Not in universe,...,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,0,0,94,- 50000.


### Notes
* "Not in universe" used; a distinct category rather than "missing" data
* "?" indicates missing data
* "weight indicates relative distribution of people in general population that each record represents due to stratified sampling"
* complex decision boundary -> motivation for gradient boosted decision tree model classification task
* weighted k-means clustering for segmentation task since weights are important in identifying statistically significant segments

## Classification Task

In [14]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# preprocessing
df_clean = df.replace('?', 'NA') # handle missing values
y = df['label'].apply(lambda x:1 if '50000+' in str(x) else 0).values # prepare target values (binary)
weights = df['weight'].values # extract weights
X = df.drop(['label', 'weight'], axis=1) # prepare features

# encode categorical features as integers for xgboost
categorical = X.select_dtypes(include=['object']).columns
label_encoders = {}
for cat in categorical:
    le = LabelEncoder()
    X[cat] = le.fit_transform(X[cat].astype(str))
    label_encoders[cat] = le
    
# split data
# 80% train & validation / 20% test
X_temp, X_test, y_temp, y_test, w_temp, w_test = train_test_split(
    X, y, weights, test_size=0.20, random_state=42, stratify=y
)

# take 20% from 80% split earlier for validation (total: 60 train/20 test/20 val)
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X_temp, y_temp, w_temp, test_size=0.25, random_state=42, stratify=y_temp
)
    
# train model
# scale_pos_weight helps with class imbalance
ratio = np.sum(y == 0) / np.sum(y == 1)

classifier = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=ratio,
    random_state=42,
    use_label_encoder=False,
    early_stopping_rounds=50,
    eval_metric='logloss'
)

classifier.fit(
    X_train, y_train,
    sample_weight=w_train, 
    eval_set=[(X_val, y_val)],
    sample_weight_eval_set=[w_val],
    verbose=10
)

# validation & evaluation
print("Final Evaluation")
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred, sample_weight=w_test)) # use sample_weight in report to reflect population distribution
important_features = pd.Series(classifier.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 10 Predictive Variables for Income")
print(important_features.head(10))

[0]	validation_0-logloss:0.67708


c:\Users\dorot\anaconda3\Lib\site-packages\xgboost\callback.py:385: UserWarning: [19:40:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[10]	validation_0-logloss:0.49875
[20]	validation_0-logloss:0.41720
[30]	validation_0-logloss:0.37397
[40]	validation_0-logloss:0.34898
[50]	validation_0-logloss:0.33231
[60]	validation_0-logloss:0.32122
[70]	validation_0-logloss:0.31318
[80]	validation_0-logloss:0.30656
[90]	validation_0-logloss:0.30063
[100]	validation_0-logloss:0.29562
[110]	validation_0-logloss:0.29129
[120]	validation_0-logloss:0.28698
[130]	validation_0-logloss:0.28371
[140]	validation_0-logloss:0.28130
[150]	validation_0-logloss:0.27876
[160]	validation_0-logloss:0.27669
[170]	validation_0-logloss:0.27452
[180]	validation_0-logloss:0.27269
[190]	validation_0-logloss:0.27113
[200]	validation_0-logloss:0.26895
[210]	validation_0-logloss:0.26785
[220]	validation_0-logloss:0.26633
[230]	validation_0-logloss:0.26505
[240]	validation_0-logloss:0.26357
[250]	validation_0-logloss:0.26224
[260]	validation_0-logloss:0.26114
[270]	validation_0-logloss:0.26025
[280]	validation_0-logloss:0.25939
[290]	validation_0-logloss:0.

## Resources
https://xgboost.readthedocs.io/en/release_3.2.0/tutorials/model.html